In [43]:
import argparse
import gymnasium as gym
import numpy as np
import scipy.optimize
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import namedtuple
from functools import partial

In [ ]:
class ActorPolicyNetwork(nn.Module):
    def __init__(self,
                 input_state_features=8, 
                 num_actions=4,
                 hidden_features=128):
        
        super(ActorPolicyNetwork, self).__init__()

        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        pi = F.softmax(self.fc3(x), dim=-1)
        return pi

class CriticValueNetwork(nn.Module):
    def __init__(self, input_state_features=8, hidden_features=128):
        super(CriticValueNetwork, self).__init__()
        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, 1)  

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        value = self.fc3(x)
        return value


In [ ]:
### Memory store that allows us to sample 30,000 experiences then do multiple gradient steps on the model by sampling the environment just once
class Memory: 

    def __init__(self):

        self.states = []
        self.actions = []
        self.rewards = []
        self.next_states = []
        self.dones = []
        
    def store(self, state, action, reward, next_state, done):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.next_states.append(next_state)
        self.dones.append(done) 

    def reset(self):
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.next_states = []
        self.dones = []

    def sample(self):

        return np.array(self.states), np.array(self.actions), np.array(self.rewards), np.array(self.next_states), np.array(self.dones)

    def __len__(self):
        return len(self.states)

In [ ]:
### Helper functions to handle model weights for custom optimizer

### need to flatten weights into a vector to do weight changes

def get_flat_params_from(model):

    return torch.cat([param.data.flatten() for param in model.parameters()])

### need to copy weights back into the model
def set_flat_params_to(model, flat_params):
    """
    Take vector of parameters and copy back into model
    """

    ### Starting index in vector is 0
    prev_ind = 0
    for param in model.parameters():

        ### Check how many parameters are in this weight matrix ###
        flat_size = int(np.prod(list(param.size())))

        ### Copy in that portion ###
        param.data.copy_(flat_params[prev_ind:prev_ind + flat_size].reshape(param.size()))
        
        ### Move starting index ###
        prev_ind += flat_size

### need to store all gradients and attach to weights
def get_flat_grad_from(net):
    """
    Copy all gradients from parameters and store
    """
    return torch.cat([param.grad.flatten() for param in net.parameters()])


In [ ]:
### Find the conjugate gradient to determine step direction
# solving for H^-1 g, which is the same as solving for Hx = g
def conjugate_gradients(Avp, b, n_iters, residual_tol=1e-10):

    # continue to step until residual is less than residual_tol

    # solving Ax = b
    x = torch.zeros(b.size()) # vector of size bx1
    r = b.clone() # r = b - Ax 
    p = b.clone() # p = r, or d0 = r0
    rdotr = torch.dot(r, r) # r0^T r0, used in finding alpha and beta

    for i in range(n_iters):

        _Avp = Avp(p) # A * d_k

        alpha = rdotr / torch.dot(p, _Avp)
        
        # x_k+1 = x_k + alpha_k * p_k
        x += alpha * p

        # find new r, r_k+1 = r_k + alpha * A * d_k
        r += alpha * _Avp

        new_rdotr = torch.dot(r,r)

        beta = new_rdotr / rdotr
        
        # find new p, p_k+1 = r_k+1 + beta * p_k
        p = r + beta * p

        rdotr = new_rdotr

        # check tolerance, where rdotr = ||r||^2
        if rdotr < residual_tol: 
            break

    return x

In [ ]:
### Line search method, given the step size and direction, we now need to walk the objective function

def linesearch(model, get_loss_method, prev_params, fullstep, expected_improve_rate, max_backtracks=10, accept_ratio=0.1): 

    loss = get_loss_method().detach()

    # try bunch of steps with ratios in descending order, so 0.5**0 -> 0.5**9
    for stepfrac in 0.5 ** np.arange(max_backtracks): 

        x_new = prev_params + stepfrac * fullstep

        set_flat_params_to(model, x_new)

        new_loss = get_loss_method().detach()

        actual_improve = loss - new_loss

        expected_improve = expected_improve_rate * step_frac

        ratio = actual_improve / expected_improve

        if ratio.item() > accept_ratio and actual_improve.item() > 0:
            return True, x_new

    return False, prev_params

In [ ]:
def trpo_step(model, get_loss_method, get_kl_loss_method, max_kl_threshold=0.01, damping=0.1):

    loss = get_loss_method()

    grads = torch.autograd.grad(loss, model.parameters())

    loss_wrt_model_grad = torch.cat([grad.flatten() for grad in grads]).detach() # gradients on the model parameters which is delta f(x) or f'(x)

    # where v is our step direction, recall we are solving for Hs
    def Fvp(v):

        ### find second derivative - H, the hessian matrix  
        kl = get_kl_loss_method().mean() # returns KL divergence of new policy and old policy

        # compute gradients of kl wrt to theta, how will nudging each model parameters alter kl, with a gradient attached to each model parameter
        grads = torch.autograd.grad(kl, model.parameters(), create_graph=True)

        # store grads as a vector
        kl_wrt_model_grad = torch.cat([grad.flatten() for grad in grads])

        ### interpret as d/dø KL * v, because autograd needs a scalar input like 'loss', we flatten it, but autograd walks the computation tree and propagates new gradients to each parameter
        kl_wrt_model_grad_v = (kl_wrt_model_grad * v).sum()
        
        ### Compute Second Derivative whcih is d/dø (d/dø KL * v) = d^2/dø^2 (KL) * v which is Hv! ###
        kl_wrt_model_twice_grad_v = torch.autograd.grad(kl_wrt_model_grad_v, model.parameters())

        ### Store Grads as Vector which gives us Hv ###
        flat_twice_grad = torch.cat([grad.flatten() for grad in kl_wrt_model_twice_grad_v]).detach()

        ### Dampening (numerical stability) ###
        flat_twice_grad = flat_twice_grad + v * damping

        return flat_twice_grad

    ### Compute the Step Direction ###
    ### We want to maximize our loss (but we are minimizing) so minimize negative loss ###
    step_direction = conjugate_gradients(Fvp, b=-loss_wrt_model_grad, n_iters=10)

    ### Now that we have our max step direction allowed by our trust region ###
    ### we can compute our KL divergence approximation 0.5 * sHs
    d_kl = 0.5 * (step_direction * Fvp(step_direction)).sum(axis=0, keepdim=True)

    ### We can Now compute Lambda ###
    lmbda = torch.sqrt(d_kl/max_kl_threshold)

    ### We can Now Compute our fullstep ###
    fullstep = step_direction / lmbda

    ### Comput our Expected Improvement for Line Search ###
    # formula for this is -objective function = -f'(x_k+1)(x_k+1 - x_k) = -f'(x_k+1)*s
    expected_improvement = (-loss_wrt_model_grad * step_direction).sum(axis=0, keepdim=True)

    ### Do Linesearch on the Parameters of the Model ###
    prev_params = get_flat_params_from(model)
    success, new_params = linesearch(model=model, 
                                     get_loss_method=get_loss_method, 
                                     prev_params=prev_params, 
                                     fullstep=fullstep,
                                     expected_improve_rate=expected_improvement)
    
    ### Copy our Linesearched parameters into our model again ###
    # this is the equivalent of our optimizer step, we've tweaked the model parameters in the step direction (which abides by the KL constraint)
    # then we used line search to find the right step size
    set_flat_params_to(model, new_params)

    return loss

In [ ]:
class Trainer: 

    def __init__(self, 
                 env,
                 num_training_iterations=250,
                 gamma=0.995, 
                 lambda_weight=0.99, 
                 weight_decay=0.001, 
                 max_kl=0.01, 
                 damping=0.1, 
                 value_optimizer="bfgs", # choices of "bfgs" or "adam"
                 adam_lr=3e-4,
                 value_optimizer_iters=25,
                 batch_size=30_000,
                 print_freq=10, 
                 running_avg_scores=25,
                 target_rewards=200):

        self.env = env
        self.iterations = num_training_iterations
        self.gamma = gamma
        self.lmbda = lambda_weight
        self.weight_decay = weight_decay
        self.max_kl = max_kl
        self.damping = damping
        self.use_adam = True if value_optimizer == "adam" else False
        self.val_optim_iters = value_optimizer_iters
        self.adam_lr = adam_lr
        self.batch_size = batch_size
        self.print_freq = print_freq
        self.target_rewards = target_rewards
        self.running_avg_scores = running_avg_scores

        self.num_inputs = env.observation_space.shape[0]
        self.num_actions = env.action_space.n
        
        self.policy_network = ActorPolicyNetwork(self.num_inputs, self.num_actions)
        self.value_network = CriticValueNetwork(self.num_inputs)

        if self.use_adam:
            self.v_optimizer = torch.optim.AdamW(self.value_network.parameters(), weight_decay=weight_decay, lr=self.adam_lr)

    def select_action(self, state):
        ### Sample Action from Policy ###
        state = torch.from_numpy(state).unsqueeze(0).to(dtype=torch.float32) # change [state] to [1, state] where state is a [1x8] vector for lunar lander
        probs = self.policy_network(state) # output 4 possible action probs
        action = torch.multinomial(probs, 1)
        return action.item(), probs

    def update_values_adam_optimizer(self, states, returns):
        
        # update value network
        for _ in range(self.val_optim_iters):
            
            ### Zero Grad ###
            self.v_optimizer.zero_grad()
    
            ### Compute Values of States ###
            values = self.value_network(states)
    
            ### Compute MSE Loss ###
            value_loss = ((values - returns)**2).mean()
    
            ### Compute Gradients ###
            value_loss.backward()
    
            ### Update Model ###
            self.v_optimizer.step()
   
    def update_values_bfgs_optimizer(self, states, returns):

        def get_value_loss(flat_params, states, returns):

            ### Copy Parameters into Value Network ###
            set_flat_params_to(self.value_network, torch.tensor(flat_params))
    
            ### Zero Gradient (same as optimizer.zero_grad() but we have no optimizer now) ###
            for param in self.value_network.parameters():
                if param.grad is not None:
                    param.grad.data.fill_(0)
    
            ### Compute Values of the States (so we can compare to target returns) ###
            values = self.value_network(states)
    
            ### Compute the MSE between our target returns and predicted values ###
            value_loss = ((values - returns)**2).mean()
    
            ### Add Weight Decay (l2) to value_loss ###
            for param in self.value_network.parameters():
                value_loss += (param**2).sum() * self.weight_decay
            
            ### Compute Gradients for Value Network (using autograd) ###
            value_loss.backward()
    
            ### L-BFGS Requires the value of the objective function want to minimize (our value_loss) ###
            ### and the gradients of the parameters. Typically we give a function that computes the gradient ###
            ### but we can use autograd to automatically compute that gradient for us! ###
            return value_loss.data.float().numpy(), get_flat_grad_from(self.value_network).data.float().numpy()

        ### Use L-BFGS To Compute estimated parameters that minimize our loss function! ###\
        value_loss_fn = partial(get_value_loss, states=states, returns=returns)
        updated_value_paramters, f, d = scipy.optimize.fmin_l_bfgs_b(func=value_loss_fn, 
                                                                     x0=get_flat_params_from(self.value_network).float().numpy(), 
                                                                     maxiter=self.val_optim_iters)
        ### Copy Update Parameters into Model ###
        set_flat_params_to(self.value_network, torch.tensor(updated_value_paramters))
        
            
    def get_policy_loss(self, states, actions, advantages, fixed_log_prob):

        ### Compute Action Probs on Current Policy Model ###
        action_probs = self.policy_network(states)

        ### Grab the Log Probs ###
        log_probs = torch.log(action_probs.gather(1, actions.unsqueeze(1)))

        ### Compute Policy loss with importance sampling ###
        policy_loss = - advantages * torch.exp(log_probs - fixed_log_prob)

        return policy_loss.mean()

    def get_kl_loss(self, states, action_probs):

        ### Standard KL Divergence Formula ###
        old_probs = action_probs.detach()
        new_probs = self.policy_network(states)
        kl = torch.sum(old_probs * torch.log(old_probs / (new_probs + 1e-8)), dim=1, keepdim=True)
        
        return kl
        
    def update_model(self, batch):

        ### Unpack Batch and Convert to Tensor ###
        states, actions, dones, next_states, rewards = batch
        rewards = torch.tensor(rewards, dtype=torch.float32)
        dones = torch.tensor(dones, dtype=torch.float32)
        actions = torch.tensor(actions)
        states = torch.tensor(states, dtype=torch.float32)

        ### Compute Values of the States ###
        values = self.value_network(states)

        ### Compute Generalized Advantage Estimation ###
        returns = torch.zeros_like(values)
        advantages = torch.zeros_like(values)
        
        ### All of our Episodes are concatenated together, lets identify the start and end index ###
        ### for each of these episodes (episode boundaries) ###
        episode_ends = [i for i in range(len(dones)) if dones[i] == 0] + [len(dones)]
        episode_starts = [0] + [i+1 for i in episode_ends[:-1]]

        ### Compute GAE for Each Episode ###
        for start, end in zip(episode_starts, episode_ends):

            episode_rewards = rewards[start:end]
            episode_dones = dones[start:end]
            episode_values = values[start:end]

            future_rewards = 0
            future_values = 0
            future_advantages = 0
            
            for i in reversed(range(len(episode_rewards))):

                ### Get Index in the full batch to Store Results ###
                idx = start + i

                ### Compute Discounted Returns ###
                returns[idx] = episode_rewards[i] + self.gamma * future_rewards * episode_dones[i] 

                ### Compute Delta (our TD Error): delta = r + gamma * V(s_{t+1}) - V(s_t) ###
                td_error = episode_rewards[i] + self.gamma * future_values * episode_dones[i] - episode_values[i]

                ### Compute our Advantage: A_t = td_error + lambda * gamma * A_{t+1} ###
                advantages[idx] = td_error + self.gamma * self.lmbda * future_advantages * episode_dones[i]

                ### Set for Next Iteration ###
                future_rewards = returns[idx]
                future_values = episode_values[i]
                future_advantages = advantages[idx]

        ### Update Values Network ###
        value_optimizer = self.update_values_adam_optimizer if self.use_adam else self.update_values_bfgs_optimizer
        value_optimizer(states, returns)

        ### Normalize Advantages ###
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        ### Get the Probs of Actios for the different states ###
        action_probs = self.policy_network(states)

        ### Convert selected actions to Log Probs for stability ###
        fixed_log_probs = torch.log(action_probs.gather(1, actions.unsqueeze(1))).detach()

        ### Take a step with the policy network ###
        trpo_step(model=self.policy_network,
                  get_loss_method=partial(self.get_policy_loss, states=states, actions=actions, advantages=advantages, fixed_log_prob=fixed_log_probs), 
                  get_kl_loss_method=partial(self.get_kl_loss, states=states, action_probs=action_probs), 
                  max_kl_threshold=self.max_kl, 
                  damping=self.damping)
        
    def train(self):

        memory = Memory()

        log = {"scores": [], "running_avg_scores": []}

        ### We will grab a batch of episodes every iteration ###
        for i in range(self.iterations):

            ### Store the number of episodes we have played ###
            num_episodes = 0

            ### Store the reward for the whole batch (multiple games) ###
            total_reward = 0

            ### Store Total Steps Taken ###
            num_steps = 0

            ### Play Batch Size of Games ###
            while num_steps <= self.batch_size:

                ### Play a Game ###
                state, _ = self.env.reset()

                done = False

                episode_steps = 0
                episode_score = 0

                while not done:

                    ### Sample Action ###
                    action, probs = self.select_action(state)

                    ### Take Step ###
                    next_state, reward, terminal, truncate, _ = env.step(action)
                    done = terminal or truncate

                    ### Iterate ###
                    episode_steps += 1

                    ### Accumulate Rewards ###
                    episode_score += reward

                    ### Store Experience Tuple ###
                    memory.store(state, action, done, next_state, reward)

                    ### Set the state as the next state ###
                    state = next_state

                ### Iterate ###
                total_reward += episode_score
                num_steps += episode_steps 
                num_episodes += 1

                ### Log the Results for this Game ###
                log["scores"].append(episode_score)
                log["running_avg_scores"].append(np.mean(log["scores"][-self.running_avg_scores:]))
        
            ### Compute Averaged Rewards Over All Games ###
            total_reward = total_reward / num_episodes

            ### Get Batch of Data and Update Model ###
            batch = memory.sample()
            self.update_model(batch)

            ### Clear Memory for the Next Batch ###
            memory.reset()

            if i % self.print_freq == 0:
                print(f'Episode {i}, Avg Batch Rewards: {total_reward:.2f}')
                
            if total_reward > self.target_rewards:
                print(f"Acheived Reward {total_reward}, Completed Training!")
                break

        return self.policy_network, self.value_network, log

In [ ]:
env = gym.make("LunarLander-v3")                

# ### Train with BFGS ### 
print("TRAINING BFGS")
trainer = Trainer(env, value_optimizer="bfgs")
bfgs_policy, bfgs_value, bfgs_log = trainer.train()

### Train with Adam ###
print("TRAINING ADAM")
trainer = Trainer(env, value_optimize="adam")
adam_policy, adam_value, adam_log = trainer.train()

TRAINING BFGS


NameError: name 'partial' is not defined